In [ ]:
import logging
import platform
import random
import subprocess
import time
from os import getenv

import fs_structs
from dotenv import load_dotenv

from distributed_processing.async_result import gather
from distributed_processing.utils import fsclient

In [ ]:
load_dotenv()
NS_PATH = getenv("NS_PATH")
MASTER_QUEUE = getenv("MASTER_QUEUE")
LAUNCH_SERVER = False

MASTER_QUEUE

In [ ]:
# logging.getLogger("distributed_processing").setLevel(logging.DEBUG)
# logging.getLogger("distributed_processing.filesystem_connector").setLevel(logging.DEBUG)

In [ ]:
if LAUNCH_SERVER:
    PYTHON_INTERPRETER = getenv("PYTHON_INTERPRETER")
    MULTI_WORKER_SCRIPT = getenv("MULTI_WORKER_SCRIPT")
    server_process = subprocess.Popen(
        [PYTHON_INTERPRETER, MULTI_WORKER_SCRIPT],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        shell=False,
    )
    time.sleep(10)

In [ ]:
# NBVAL_CHECK_OUTPUT
"""
fs_connector = FileSystemConnector(NS_PATH)
fs_connector.with_watchdog=True
fs_connector.pop_watchdog_timeout = 10

client = Client(DummySerializer(), fs_connector, check_registry="cache")
"""

client = fsclient(NS_PATH)
client.timeout = 30

In [ ]:
MASTER_QUEUE = client.to_requests_queue(MASTER_QUEUE)

In [ ]:
client.update_registry_cache()

In [ ]:
ns = fs_structs.structs.FSNamespace(NS_PATH)

In [ ]:
# NBVAL_CHECK_OUTPUT
ns.registry.keys()

In [ ]:
z1 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z2 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z3 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)

(z1, z2, z3)

In [ ]:
ps = client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)
ps

In [ ]:
# NBVAL_CHECK_OUTPUT

len(ps)

In [ ]:
# NBVAL_CHECK_OUTPUT

[(w, q) for _, w, q in ps]

In [ ]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync("kill_process", [z2], queue=MASTER_QUEUE)

In [ ]:
client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)

In [ ]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

In [ ]:
client.rpc_sync("kill_all_processes", [], queue=MASTER_QUEUE)

In [ ]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

In [ ]:
z1 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z2 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z3 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)

In [ ]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

In [ ]:
client.update_registry_cache()

In [ ]:
y = client.rpc_async("add", [1, 0])

In [ ]:
# NBVAL_CHECK_OUTPUT

y.get()

In [ ]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"

try:
    y = client.rpc_async("fake", [1, 0])
except Exception as e:
    print(e)

In [ ]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"  # use queue client.default_queue, by default "default"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

try:
    y.get()
except Exception as e:
    print(e)


In [ ]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

y.safe_get(default="Esto es un error")

In [ ]:
client.check_registry = "cache"

In [ ]:
x = client.rpc_async("div", [1, 0])

In [ ]:
try:
    x.get()
except Exception as e:
    print(e)

In [ ]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print("Error")

In [ ]:
# x = client.rpc_sync("div", [1, 0])

In [ ]:
# NBVAL_CHECK_OUTPUT


client.rpc_sync("add", [1, 1])

In [ ]:
def f(x, y):
    return x + y


y = client.rpc_async_fn(f, [1, 2.0])

In [ ]:
y.get()

In [ ]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync_fn(f, [3.0, 2.0])

In [ ]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "lista", "tupla", "dic"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

t = ("sleep", [10], {})
tp.append(t)
tp


In [ ]:
tp = [
    ("div", [0.5273328219558507, 0.13835718442890943], {}),
    ("div", [0.7224042937278776, 0.6744424742714074], {}),
    ("mul", [0.16464192867054384, 0.2961055537758295], {}),
    ("lista", [0.24324779336838243, 0.4486736957376778], {}),
    ("dic", [0.222443603731179, 0.049201002693669005], {}),
    ("dic", [0.5892562777042863, 0.8190093828367946], {}),
    ("div", [0.9698052964451762, 0.2495544466990297], {}),
    ("add", [0.18281717945238662, 0.28456253134154685], {}),
    ("div", [0.8231058337900704, 0.4550312056693214], {}),
    ("mul", [0.6955981505190975, 0.2960636895338262], {}),
    ("add", [0.5793774647414703, 0.9353212122782703], {}),
    ("lista", [0.3893530442489298, 0.74231052966268], {}),
    ("div", [0.6419882052325951, 0.7661892480993476], {}),
    ("div", [0.049994104986677, 0.4562378471461709], {}),
    ("dic", [0.4734342157728231, 0.053714834674179035], {}),
    ("mul", [0.8092977961625194, 0.9195146847049076], {}),
    ("mul", [0.913778636227633, 0.6145608354175943], {}),
    ("lista", [0.7499808955353686, 0.5337360859450593], {}),
    ("dic", [0.6756209555432302, 0.9505296005797351], {}),
    ("dic", [0.5209295316393681, 0.9420323858687962], {}),
    ("div", [0.15809810362813237, 0.62619392590883], {}),
    ("dic", [0.29474022335742067, 0.893515494048087], {}),
    ("mul", [0.5349022233942071, 0.05757455715844495], {}),
    ("lista", [0.042102069906052586, 0.3469740971990074], {}),
    ("mul", [0.871021663889528, 0.007201521377221853], {}),
    ("lista", [0.6049724123450363, 0.26801461728942155], {}),
    ("dic", [0.8898534023208522, 0.5019296231298458], {}),
    ("lista", [0.9266421130454301, 0.9972178178045238], {}),
    ("mul", [0.48602513421859184, 0.199817263488683], {}),
    ("dic", [0.7163791721275343, 0.6950721561324937], {}),
    ("sleep", [10], {}),
]

In [ ]:
fs = []
for t in tp:
    fs.append(client.rpc_async(t[0], t[1], t[2]))

In [ ]:
# NBVAL_CHECK_OUTPUT

resultados = [f.get() for f in fs]
resultados

In [ ]:
# NBVAL_CHECK_OUTPUT

fs = client.rpc_multi_async(tp)

In [ ]:
time.sleep(3)

In [ ]:
# NBVAL_CHECK_OUTPUT

# AsynResult.status updates the information every time it is called.
# 'PENDING' state should be assumed as transitory.
# If there are responses available in the queue or in the cache
# status or pending() updates the AsyncResult object.

[f.status for f in fs]

In [ ]:
r = client.wait_responses(timeout=2)
r

In [ ]:
# NBVAL_CHECK_OUTPUT

len(r)

In [ ]:
time.sleep(7)

In [ ]:
# NBVAL_CHECK_OUTPUT
# AsynResult.status updates information

[f.status for f in fs]

In [ ]:
# NBVAL_CHECK_OUTPUT

client.wait_responses()

In [ ]:
# NBVAL_CHECK_OUTPUT

try:
    client.wait_responses(["kk", "qq"])
except ValueError as e:
    print(e)

In [ ]:
client._update_cache_with_all_available_responses()

In [ ]:
# NBVAL_CHECK_OUTPUT

client.pending

In [ ]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

In [ ]:
fs = client.rpc_batch_async(tp)

In [ ]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

In [ ]:
# NBVAL_CHECK_OUTPUT

client.rpc_batch_sync(tp)

In [ ]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "fake"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

tp

In [ ]:
tp = [
    ("mul", [0.7355407026565478, 0.023519777553893007], {}),
    ("fake", [0.2520522260048764, 0.28054227055242864], {}),
    ("mul", [0.8838106264839363, 0.19639888038506714], {}),
    ("mul", [0.8857412900943293, 0.1253016179972587], {}),
    ("add", [0.0005226378683395039, 0.6568308199617323], {}),
    ("add", [0.15476536347494607, 0.4492869825171584], {}),
    ("fake", [0.7631067253940333, 0.019049359538004906], {}),
    ("fake", [0.4310102131915736, 0.675491507770126], {}),
    ("fake", [0.7140511506543913, 0.7837833237953286], {}),
    ("add", [0.0909525538014786, 0.44959184616881276], {}),
    ("add", [0.6627327150574825, 0.7401973301261011], {}),
    ("div", [0.21232540669537237, 0.7667374101737603], {}),
    ("add", [0.887254961441083, 0.21290364755712576], {}),
    ("fake", [0.7372649371990656, 0.3796617846297834], {}),
    ("add", [0.30864649241428155, 0.8777033159855755], {}),
    ("div", [0.4997997676145608, 0.45884184399530026], {}),
    ("fake", [0.0011733893324340494, 0.6016126834507816], {}),
    ("mul", [0.9307150630961242, 0.2943025202085412], {}),
    ("div", [0.6545197868355805, 0.11276464241684814], {}),
    ("mul", [0.6573680105483782, 0.13653818666573825], {}),
    ("add", [0.7959397704608381, 0.41576468218269147], {}),
    ("mul", [0.16347738503061415, 0.04898545483226813], {}),
    ("fake", [0.7815677830141265, 0.013945854620984632], {}),
    ("fake", [0.8020187012792446, 0.25875661742111045], {}),
    ("fake", [0.9043915109893543, 0.6876522434184933], {}),
    ("add", [0.929922781635924, 0.9540258004221696], {}),
    ("fake", [0.961238888823737, 0.4030318469598062], {}),
    ("div", [0.7466755564012741, 0.799944178434153], {}),
    ("mul", [0.15308290555092918, 0.45297088397324015], {}),
    ("mul", [0.3505532310800745, 0.5551384076337974], {}),
]

In [ ]:
client.check_registry = "never"
client.set_default_queue("cola_1")

fs = client.rpc_batch_async(tp)

In [ ]:
# NBVAL_CHECK_OUTPUT

[f.safe_get() for f in fs]

In [ ]:
# NBVAL_CHECK_OUTPUT

client.responses

In [ ]:
client.check_registry = "cache"

In [ ]:
# NBVAL_CHECK_OUTPUT

try:
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

In [ ]:
client.check_registry = "never"
client.set_default_queue("cola_1")

x = client.rpc_async("kk", [1, 0])

In [ ]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print(e)

In [ ]:
y = client.rpc_async("add", [1, 0])

In [ ]:
# NBVAL_CHECK_OUTPUT

y.get(5)

In [ ]:
# NBVAL_CHECK_OUTPUT


def f(x, y):
    return x + y


# "never" use queue "default" don't work for rpc_async_fn or rpc_sync_fn
client.check_registry = "never"
y = client.rpc_async_fn(f, [1, 2.0])
try:
    print(y.get())
except Exception as e:
    print(e)

In [ ]:
client.check_registry = "cache"
y = client.rpc_async_fn(f, [1, 2.0])

In [ ]:
# NBVAL_CHECK_OUTPUT

y.get()

In [ ]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

In [ ]:
client.clean_used()

In [ ]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

In [ ]:
# client.rpc_sync_fn(print, ["hola"])

In [ ]:
# NBVAL_CHECK_OUTPUT

tp = [
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
]
tp

In [ ]:
# NBVAL_CHECK_OUTPUT

client.default_requests_queue

In [ ]:
client.check_registry = "never"
fs = client.rpc_multi_async(tp, retry=True)

In [ ]:
# NBVAL_CHECK_OUTPUT

gather(fs, 30, 5)

In [ ]:
s = [f.status for f in fs]

In [ ]:
# NBVAL_CHECK_OUTPUT

len(s), "OK" in s

In [ ]:
[
    (time.time() if f.finished_time is None else f.finished_time) - f.creation_time
    for f in fs
]

In [ ]:
[f.finished_time for f in fs]

In [ ]:
client.pending

In [ ]:
fs[3].status

In [ ]:
fs[4].retry()

In [ ]:
# [f.retry() for f in fs if not f.done()]
# no hace falta chequear si está pendiente.
[f.retry() for f in fs]

In [ ]:
client.check_registry = "always"

In [ ]:
# NBVAL_CHECK_OUTPUT

sorted(client.connector.all_queues_for_method("info"))

In [ ]:
client.update_registry_cache()

In [ ]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "Never"
client.all_queues_for_method("hola")

In [ ]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"
a = client.rpc_async("info")

In [ ]:
# client.rpc_sync("div", [2, 1], timeout=10)

In [ ]:
x = a.get()
x

In [ ]:
# NBVAL_CHECK_OUTPUT

len(x), len(x[1])

In [ ]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_cola_1"]

In [ ]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_cola_2"]

In [ ]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_py_eval"]

In [ ]:
# NBVAL_CHECK_OUTPUT

client.default_requests_queue

In [ ]:
client.set_default_queue("cola_1")

In [ ]:
# NBVAL_CHECK_OUTPUT

client.default_requests_queue

In [ ]:
if LAUNCH_SERVER:
    client.rpc_sync("kill_all_processes", [], queue=MASTER_QUEUE, timeout=120)
    if platform.system() == "Windows":
        subprocess.run(
            ["taskkill", "/F", "/T", "/PID", str(server_process.pid)],
            capture_output=True,
        )

    else:
        server_process.terminate()
        server_process.wait()

In [ ]:
# NBVAL_CHECK_OUTPUT

ps = client.rpc_async("list_processes", [], queue=MASTER_QUEUE)
ps.safe_get(3)

In [ ]:
# NBVAL_CHECK_OUTPUT
try:
    y = client.rpc_async("add", [1, 0])
    y.safe_get(3)
except KeyError:
    print("Error")